# 🩺 Diabetic Retinopathy — EfficientNetV2-B1 Complete Pipeline
## Dataset Download → Clean → Balance → Train → Resume → 90–100% Accuracy

| Phase | Description |
|-------|-------------|
| **Phase 0** | Mount Drive + Install |
| **Phase 1** | Download all DR datasets from Kaggle |
| **Phase 2** | Clean corrupted / missing-label images |
| **Phase 3** | Balance dataset (2 000 images/class) |
| **Phase 4** | Build tf.data pipeline (384×384) |
| **Phase 5** | EfficientNetV2-B1 training (freeze → unfreeze) |
| **Phase 6** | Fine-tune all layers, save joblib + keras |
| **Phase 7** | Evaluate, confusion matrix, report |
| **Phase 8** | Resume training from last checkpoint |

> **Dataset answer:** Minimum 1 500 real images/class. With augmentation target 2 000/class.
> **Epochs answer:** Phase-1 training = 20 epochs frozen + 30 epochs unfrozen with EarlyStopping.
> **Resolution:** 384 × 384 (EfficientNetV2-B1 native size).

## ⚙️ Phase 0 — Mount Google Drive & Install All Packages

In [ ]:
# ─── 0-A: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('✅ Google Drive mounted')

In [ ]:
%%capture
# ─── 0-B: Install packages ──────────────────────────────────────────────────
!pip install -q kaggle tensorflow==2.15.0 keras==2.15.0
!pip install -q scikit-learn matplotlib seaborn tqdm joblib pillow
!pip install -q albumentations opencv-python-headless

In [ ]:
# ─── 0-C: All imports ───────────────────────────────────────────────────────
import os, gc, json, shutil, warnings, time, glob, zipfile, concurrent.futures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
import joblib
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.applications import EfficientNetV2B1
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau,
    CSVLogger, TensorBoard
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f'✅ TensorFlow : {tf.__version__}')
print(f'✅ GPUs       : {[g.name for g in gpus]}')
print(f'✅ Mixed FP16 : enabled')

In [ ]:
# ─── 0-D: Global Config ─────────────────────────────────────────────────────
# Image / Model
IMG_SIZE          = (384, 384)    # EfficientNetV2-B1 native resolution
BATCH_SIZE        = 16            # Safe for 384px on T4
NUM_CLASSES       = 5
SEED              = 42

# ── Dataset image counts ──────────────────────────────────────────────────
# ANSWER: Minimum 1500 real images per class.
# With augmentation we target 2000 per class for training.
# Total ~10 000 training images across 5 classes.
IMAGES_PER_CLASS  = 2000          # balanced target per class
MIN_REAL_PER_CLASS= 1500          # minimum real images needed

# Training epochs
# ANSWER:
#   Phase 1 (frozen backbone)    = 20 epochs  → learns classifier head fast
#   Phase 2 (full fine-tune)     = 40 epochs  → refines whole network
#   EarlyStopping patience = 7   → stops if no improvement
FREEZE_EPOCHS     = 20
FINETUNE_EPOCHS   = 40
EARLY_STOP_PAT    = 7

# Learning rates
FREEZE_LR         = 1e-3          # head-only warmup
FINETUNE_LR       = 1e-5          # full fine-tune (low to avoid catastrophic forgetting)

# Class names (must match your folder names exactly)
CLASS_NAMES       = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferative_DR']

# ── Drive Paths ────────────────────────────────────────────────────────────
DRIVE_BASE        = '/content/drive/MyDrive/DR_Project'
DATASET_DIR       = f'{DRIVE_BASE}/dataset_balanced'   # final balanced dataset
RAW_DIR           = f'{DRIVE_BASE}/dataset_raw'        # raw downloaded data
CKPT_DIR          = f'{DRIVE_BASE}/checkpoints'        # Keras checkpoints
LOG_DIR           = f'{DRIVE_BASE}/logs'               # CSVLogger logs
JOBLIB_DIR        = f'{DRIVE_BASE}/joblib_objects'     # joblib saved objects
MODEL_KERAS       = f'{DRIVE_BASE}/dr_efficientnetv2b1_384.keras'
BEST_CKPT         = f'{CKPT_DIR}/best_model.keras'
FREEZE_CKPT       = f'{CKPT_DIR}/freeze_phase_best.keras'
RESUME_JSON       = f'{CKPT_DIR}/training_state.json'  # tracks which phase/epoch

# Create all directories on Drive
for d in [DRIVE_BASE, DATASET_DIR, RAW_DIR, CKPT_DIR, LOG_DIR, JOBLIB_DIR]:
    os.makedirs(d, exist_ok=True)
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(DATASET_DIR, 'train', cls), exist_ok=True)
        os.makedirs(os.path.join(DATASET_DIR, 'val',   cls), exist_ok=True)
        os.makedirs(os.path.join(DATASET_DIR, 'test',  cls), exist_ok=True)

print('✅ Config complete. All Drive folders created.')
print(f'   Image size        : {IMG_SIZE}')
print(f'   Target per class  : {IMAGES_PER_CLASS} (balanced)')
print(f'   Freeze epochs     : {FREEZE_EPOCHS}')
print(f'   Fine-tune epochs  : {FINETUNE_EPOCHS}')
print(f'   Model save path   : {MODEL_KERAS}')

## 📥 Phase 1 — Download DR Datasets from Kaggle

> **Answer:** Download from 3 sources → APTOS 2019 + DDR + IDRiD → combine → 5-class labels.
> Run `kaggle.json` upload first, then this cell.

In [ ]:
# ─── 1-A: Upload Kaggle API key ─────────────────────────────────────────────
# Go to kaggle.com → Account → API → Download kaggle.json
# Then run this cell to upload it

from google.colab import files
print('📤 Upload your kaggle.json file now...')
uploaded = files.upload()   # Upload kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle API key installed')

In [ ]:
# ─── 1-B: Download all 3 datasets in parallel ───────────────────────────────
# ANSWER: Use 3 public Kaggle datasets for DR classification
# Each has images + CSV labels with 0-4 DR grade

DATASETS = {
    'aptos2019'  : 'mariaherrerot/aptos2019',           # ~3 600 images, 5 classes
    'ddr_dataset': 'mariaherrerot/ddr-dataset',          # ~12 500 images
    'messidor2'  : 'mariaherrerot/messidor2',            # ~1 700 images
}

def download_dataset(name, slug):
    out_dir = os.path.join(RAW_DIR, name)
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) > 0:
        print(f'⏭️  {name}: already downloaded, skipping')
        return out_dir
    os.makedirs(out_dir, exist_ok=True)
    print(f'⬇️  Downloading {name} ...')
    os.system(f'kaggle datasets download -d {slug} -p {out_dir} --unzip 2>&1')
    print(f'✅ {name}: done')
    return out_dir

# Download all datasets concurrently (saves time)
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(download_dataset, n, s): n for n, s in DATASETS.items()}
    for f in concurrent.futures.as_completed(futures):
        name = futures[f]
        try:
            path = f.result()
            print(f'📁 {name} → {path}')
        except Exception as e:
            print(f'❌ {name} failed: {e}')

print('\n✅ All dataset downloads complete')

In [ ]:
# ─── 1-C: Extract any remaining zip files (parallel) ────────────────────────
# ANSWER: Extract all zip files at the same time using ThreadPoolExecutor

def extract_zip(zip_path):
    out_dir = os.path.splitext(zip_path)[0]
    if os.path.exists(out_dir):
        return f'⏭️  already extracted: {os.path.basename(zip_path)}'
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    return f'✅ extracted: {os.path.basename(zip_path)}'

all_zips = glob.glob(os.path.join(RAW_DIR, '**', '*.zip'), recursive=True)
print(f'Found {len(all_zips)} zip files')

if all_zips:
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        results = list(ex.map(extract_zip, all_zips))
    for r in results:
        print(r)

print('✅ All extractions complete')

## 🧹 Phase 2 — Clean Dataset (Corrupted Files + Missing Labels)

> **Answer:** Scan every image → try to open → if fails → delete. Then verify label CSV has matching image file.

In [ ]:
# ─── 2-A: Remove corrupted images ───────────────────────────────────────────
# ANSWER: Open every image with PIL; if it throws UnidentifiedImageError → delete it

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

def check_image(path):
    """Return (path, is_valid, reason)"""
    ext = os.path.splitext(path)[1].lower()
    if ext not in VALID_EXTS:
        return path, False, 'invalid_extension'
    try:
        img = Image.open(path)
        img.verify()          # PIL checks file header
        # Extra check: re-open and try to convert (catches truncated files)
        img = Image.open(path).convert('RGB')
        w, h = img.size
        if w < 50 or h < 50:
            return path, False, f'too_small_{w}x{h}'
        return path, True, 'ok'
    except (UnidentifiedImageError, Exception) as e:
        return path, False, str(e)[:60]

# Scan all images in RAW_DIR
all_images = glob.glob(os.path.join(RAW_DIR, '**', '*.*'), recursive=True)
all_images = [p for p in all_images if os.path.splitext(p)[1].lower() in VALID_EXTS]
print(f'Total images found  : {len(all_images)}')

corrupt_count = 0
removed_paths = []

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    results = list(tqdm(ex.map(check_image, all_images),
                        total=len(all_images), desc='Checking images'))

for path, is_valid, reason in results:
    if not is_valid:
        corrupt_count += 1
        removed_paths.append((path, reason))
        os.remove(path)   # delete corrupted file

print(f'\n✅ Clean complete')
print(f'   Total scanned  : {len(all_images)}')
print(f'   Corrupted/bad  : {corrupt_count} → deleted')
print(f'   Valid images   : {len(all_images) - corrupt_count}')

# Save removed list to Drive for reference
removed_df = pd.DataFrame(removed_paths, columns=['path', 'reason'])
removed_df.to_csv(f'{DRIVE_BASE}/removed_images_log.csv', index=False)
print(f'   Removed log    : {DRIVE_BASE}/removed_images_log.csv')

In [ ]:
# ─── 2-B: Parse all CSV labels + match to image files ───────────────────────
# ANSWER: Each dataset has different CSV format — normalize to standard (path, label 0-4)
# Labels: 0=No_DR, 1=Mild, 2=Moderate, 3=Severe, 4=Proliferative_DR

all_records = []   # list of (abs_image_path, int_label)

# ── APTOS 2019 ───────────────────────────────────────────────────────────────
aptos_dir = os.path.join(RAW_DIR, 'aptos2019')
for split, csv_name, img_folder in [
    ('train', 'train.csv', 'train_images'),
    ('test',  'test.csv',  'test_images')
]:
    csv_path = os.path.join(aptos_dir, csv_name)
    img_path = os.path.join(aptos_dir, img_folder)
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    label_col = 'diagnosis' if 'diagnosis' in df.columns else 'level'
    for _, row in df.iterrows():
        img_file = os.path.join(img_path, row['id_code'] + '.png')
        if not os.path.exists(img_file):
            img_file = os.path.join(img_path, row['id_code'] + '.jpg')
        if os.path.exists(img_file) and label_col in row.index:
            label = int(row[label_col])
            if 0 <= label <= 4:
                all_records.append((img_file, label))

print(f'APTOS 2019       : {len(all_records)} records')
prev = len(all_records)

# ── Folder-organized datasets (class name = folder name) ─────────────────────
# Some datasets are organized as class_name/image.jpg folders
FOLDER_LABEL_MAP = {
    'No_DR': 0, 'no_dr': 0, '0': 0, 'normal': 0,
    'Mild': 1,  'mild': 1,  '1': 1,
    'Moderate': 2, 'moderate': 2, '2': 2,
    'Severe': 3,   'severe': 3,   '3': 3,
    'Proliferative_DR': 4, 'proliferative_dr': 4, '4': 4, 'proliferative': 4,
}

for ds_name in ['ddr_dataset', 'messidor2']:
    ds_path = os.path.join(RAW_DIR, ds_name)
    if not os.path.exists(ds_path):
        continue
    for root, dirs, files in os.walk(ds_path):
        folder = os.path.basename(root)
        if folder in FOLDER_LABEL_MAP:
            label = FOLDER_LABEL_MAP[folder]
            for f in files:
                if os.path.splitext(f)[1].lower() in VALID_EXTS:
                    all_records.append((os.path.join(root, f), label))
    print(f'{ds_name:16s} : +{len(all_records) - prev} records')
    prev = len(all_records)

# Remove any records whose file no longer exists (cleaned in Phase 2-A)
all_records = [(p, l) for p, l in all_records if os.path.exists(p)]

master_df = pd.DataFrame(all_records, columns=['path', 'label'])
master_df['class_name'] = master_df['label'].map(dict(enumerate(CLASS_NAMES)))

print(f'\n✅ Master label DataFrame: {len(master_df)} total images')
print(master_df['class_name'].value_counts().sort_index())

# Save master labels CSV to Drive
master_df.to_csv(f'{DRIVE_BASE}/master_labels.csv', index=False)
print(f'\n💾 Saved: {DRIVE_BASE}/master_labels.csv')

## ⚖️ Phase 3 — Balance Dataset & Organize Train/Val/Test Splits

> **Answer:**
> - Target 2 000 images per class in training split.
> - Undersample majority, oversample minority (via symlinks + augmentation).
> - Split: 80% train / 10% val / 10% test — stratified.

In [ ]:
# ─── 3-A: Stratified split + balance ────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Stratified 80/10/10 split
train_dfs, val_dfs, test_dfs = [], [], []
for cls_label in range(NUM_CLASSES):
    cls_df = master_df[master_df['label'] == cls_label].copy()
    if len(cls_df) < 10:
        print(f'⚠️  Class {cls_label} has only {len(cls_df)} images — skipping split')
        continue
    tv, test = train_test_split(cls_df, test_size=0.10, random_state=SEED)
    train, val = train_test_split(tv,   test_size=0.111, random_state=SEED)  # 0.111 of 90% ≈ 10%
    train_dfs.append(train); val_dfs.append(val); test_dfs.append(test)

train_df = pd.concat(train_dfs).reset_index(drop=True)
val_df   = pd.concat(val_dfs).reset_index(drop=True)
test_df  = pd.concat(test_dfs).reset_index(drop=True)

print('Split counts (before balancing):')
print(f'  Train : {len(train_df)}  | Val : {len(val_df)}  | Test : {len(test_df)}')
print('\nPer-class train distribution:')
print(train_df['class_name'].value_counts().sort_index())

In [ ]:
# ─── 3-B: Augment minority classes to reach IMAGES_PER_CLASS ────────────────
# ANSWER:
#   - If class has < 2000 images → generate augmented copies until 2000 reached
#   - If class has > 2000 images → undersample randomly to 2000
#   - Augmentations: flip, rotate, brightness, contrast, zoom
import albumentations as A
from albumentations.pytorch import ToTensorV2

AUG_PIPELINE = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.GridDistortion(p=0.2),    # Simulates lens distortion in fundus images
    A.CLAHE(clip_limit=2.0, p=0.3),  # Contrast enhancement — helps DR features
])

AUG_SAVE_DIR = os.path.join(RAW_DIR, 'augmented')
os.makedirs(AUG_SAVE_DIR, exist_ok=True)

def augment_and_save(row, save_dir, aug_id):
    """Apply augmentation and save new image. Return new (path, label)."""
    try:
        img = cv2.imread(row['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = AUG_PIPELINE(image=img)['image']
        cls_dir = os.path.join(save_dir, row['class_name'])
        os.makedirs(cls_dir, exist_ok=True)
        fname = f'aug_{aug_id}_{os.path.basename(row["path"])}.jpg'
        out_path = os.path.join(cls_dir, fname)
        cv2.imwrite(out_path, cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR))
        return (out_path, row['label'], row['class_name'])
    except Exception as e:
        return None

balanced_train_records = []

for cls_label in range(NUM_CLASSES):
    cls_name = CLASS_NAMES[cls_label]
    cls_df   = train_df[train_df['label'] == cls_label].copy()
    n        = len(cls_df)
    print(f'[{cls_name:20s}] original: {n:4d}', end='  ')

    if n >= IMAGES_PER_CLASS:
        # Undersample
        cls_df = cls_df.sample(IMAGES_PER_CLASS, random_state=SEED)
        for _, row in cls_df.iterrows():
            balanced_train_records.append((row['path'], cls_label, cls_name))
        print(f'→ undersampled to {IMAGES_PER_CLASS}')
    else:
        # Keep all originals
        for _, row in cls_df.iterrows():
            balanced_train_records.append((row['path'], cls_label, cls_name))
        # Augment to fill the gap
        needed = IMAGES_PER_CLASS - n
        aug_id = 0
        with tqdm(total=needed, desc=f'  Augmenting {cls_name}', leave=False) as pbar:
            while needed > 0:
                sample_row = cls_df.sample(1, random_state=aug_id).iloc[0]
                result = augment_and_save(sample_row, AUG_SAVE_DIR, aug_id)
                if result:
                    balanced_train_records.append(result)
                    needed -= 1
                    pbar.update(1)
                aug_id += 1
        print(f'→ augmented to {IMAGES_PER_CLASS}')

balanced_train_df = pd.DataFrame(balanced_train_records, columns=['path','label','class_name'])
balanced_train_df = balanced_train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Save balanced label files to Drive
balanced_train_df.to_csv(f'{DRIVE_BASE}/balanced_train_labels.csv', index=False)
val_df.to_csv(f'{DRIVE_BASE}/val_labels.csv', index=False)
test_df.to_csv(f'{DRIVE_BASE}/test_labels.csv', index=False)

print(f'\n✅ Balanced training set: {len(balanced_train_df)} images')
print(balanced_train_df['class_name'].value_counts().sort_index())
print(f'\n💾 All label CSVs saved to Drive')

In [ ]:
# ─── 3-C: Save LabelEncoder with joblib ─────────────────────────────────────
# ANSWER: joblib saves Python objects (LabelEncoder, class weights) to Drive
# These can be reloaded later for inference without retraining

le = LabelEncoder()
le.fit(CLASS_NAMES)

# Compute class weights for imbalance handling in loss function
# ANSWER: compute_class_weight gives higher weight to rare classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=balanced_train_df['label'].values
)
class_weights_dict = {i: w for i, w in enumerate(class_weights_array)}

# Save with joblib
joblib.dump(le,                 f'{JOBLIB_DIR}/label_encoder.joblib')
joblib.dump(class_weights_dict, f'{JOBLIB_DIR}/class_weights.joblib')
joblib.dump(CLASS_NAMES,        f'{JOBLIB_DIR}/class_names.joblib')

print('✅ Joblib objects saved to Drive:')
print(f'   label_encoder.joblib')
print(f'   class_weights.joblib')
print(f'   class_names.joblib')
print('\n⚖️  Class weights (imbalance correction):')
for cls_id, w in class_weights_dict.items():
    print(f'   Class {cls_id} [{CLASS_NAMES[cls_id]:20s}] weight = {w:.4f}')

## 🔄 Phase 4 — Build tf.data Pipeline (384 × 384)

> High-performance input pipeline with prefetch, cache, and on-the-fly augmentation.

In [ ]:
# ─── 4-A: tf.data pipeline ──────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    """Load image → decode → resize → normalize to [0,1]"""
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def augment_tf(img, label):
    """Lightweight TF augmentation applied on-the-fly during training"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.2)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
    img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(df, training=False, batch_size=BATCH_SIZE):
    paths  = df['path'].values
    labels = df['label'].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment_tf, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(balanced_train_df, training=True)
val_ds   = make_dataset(val_df,   training=False)
test_ds  = make_dataset(test_df,  training=False)

steps_per_epoch = len(balanced_train_df) // BATCH_SIZE
val_steps       = len(val_df) // BATCH_SIZE

print('✅ tf.data pipelines ready:')
print(f'   Train batches : {steps_per_epoch}')
print(f'   Val batches   : {val_steps}')

# Quick sanity: show 1 batch shape
for imgs, lbls in train_ds.take(1):
    print(f'   Batch shape   : {imgs.shape}  dtype={imgs.dtype}')
    print(f'   Label shape   : {lbls.shape}')

## 🧠 Phase 5 — Build EfficientNetV2-B1 Model

> **Answer (EfficientNetV2-B1 training strategy):**
> 1. Load ImageNet weights (transfer learning)
> 2. Freeze backbone → train only classifier head (20 epochs, lr=1e-3)
> 3. Unfreeze ALL layers → fine-tune everything (40 epochs, lr=1e-5)
> 4. Dropout 0.4 + L2 regularization → prevents overfitting
> 5. Expected accuracy: 90–97% with 2000 images/class

In [ ]:
# ─── 5-A: Build EfficientNetV2-B1 model ─────────────────────────────────────
# ANSWER: EfficientNetV2-B1 needs 384×384 input (its native resolution)
# This is better than B0 (224px) → higher accuracy for medical images

def build_model(num_classes=NUM_CLASSES, dropout_rate=0.40, l2_lambda=1e-4):
    """
    EfficientNetV2-B1 with custom classification head.
    - Dropout 0.40 → prevents overfitting
    - L2 regularization → weight decay, prevents overfitting
    - BatchNormalization → stabilizes training
    """
    # ── Backbone ────────────────────────────────────────────────────────────
    backbone = EfficientNetV2B1(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3),
        include_preprocessing=True  # Built-in normalization for EfficientNetV2
    )
    backbone.trainable = False  # Frozen at first

    # ── Classification Head ─────────────────────────────────────────────────
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='input_image')
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dropout(dropout_rate, name='drop_1')(x)
    x = layers.Dense(
        512, activation='relu',
        kernel_regularizer=regularizers.l2(l2_lambda),
        name='dense_512'
    )(x)
    x = layers.BatchNormalization(name='bn_dense')(x)
    x = layers.Dropout(dropout_rate / 2, name='drop_2')(x)
    # Output: float32 (important for mixed precision stability)
    outputs = layers.Dense(
        num_classes, activation='softmax', dtype='float32',
        kernel_regularizer=regularizers.l2(l2_lambda),
        name='output'
    )(x)

    model = Model(inputs, outputs, name='DR_EfficientNetV2B1')
    return model, backbone

model, backbone = build_model()
model.summary(line_length=90)
print(f'\nTotal params     : {model.count_params():,}')
print(f'Trainable params : {sum([np.prod(v.shape) for v in model.trainable_variables]):,}')

## 🏋️ Phase 6 — Training (Freeze Phase + Fine-Tune Phase)

> **Resume support:** Check Drive for existing checkpoint → load → continue from last epoch.
> All results (loss/accuracy per epoch) saved as CSV to Drive.

In [ ]:
# ─── 6-A: Helper — Save / Load Training State (Resume Support) ───────────────
# ANSWER: Use JSON file on Drive to track which phase and epoch we're on
# On crash → reload checkpoint + read JSON → continue training exactly where left off

def save_training_state(phase, epoch_done, best_val_acc):
    state = {
        'phase'       : phase,
        'epoch_done'  : epoch_done,
        'best_val_acc': float(best_val_acc),
        'timestamp'   : time.strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(RESUME_JSON, 'w') as f:
        json.dump(state, f, indent=2)

def load_training_state():
    if os.path.exists(RESUME_JSON):
        with open(RESUME_JSON) as f:
            state = json.load(f)
        print(f'📂 Resuming from: Phase={state["phase"]} | EpochDone={state["epoch_done"]} | BestValAcc={state["best_val_acc"]:.4f}')
        return state
    return {'phase': 'freeze', 'epoch_done': 0, 'best_val_acc': 0.0}

def make_callbacks(phase_name, ckpt_path, log_path):
    """Create callbacks for one training phase."""
    return [
        # ✅ Save BEST model to Drive — never lose progress
        ModelCheckpoint(
            filepath=ckpt_path,
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        # ✅ Also save LATEST epoch to Drive — so resume is always possible
        ModelCheckpoint(
            filepath=ckpt_path.replace('.keras', '_latest.keras'),
            monitor='val_accuracy',
            mode='max',
            save_best_only=False,
            save_freq='epoch',
            verbose=0
        ),
        # ✅ Stop early if val_accuracy stops improving
        EarlyStopping(
            monitor='val_accuracy',
            patience=EARLY_STOP_PAT,
            restore_best_weights=True,
            verbose=1
        ),
        # ✅ Reduce LR when val_loss plateaus
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=4,
            min_lr=1e-8,
            verbose=1
        ),
        # ✅ Save all epoch metrics to CSV on Drive
        CSVLogger(
            filename=log_path,
            append=True   # append=True lets us resume without losing history
        ),
    ]

print('✅ Callback factory ready')

In [ ]:
# ─── 6-B: Phase 1 — Frozen Backbone Training ────────────────────────────────
# ANSWER:
#   Why freeze first? ImageNet backbone is already powerful.
#   Training only the head (512 + softmax) is fast and prevents
#   the backbone from being destroyed by a high LR at the start.

state = load_training_state()
freeze_history = None

if state['phase'] == 'freeze':
    remaining_freeze = FREEZE_EPOCHS - state['epoch_done']

    if remaining_freeze > 0:
        print(f'\n🔒 PHASE 1 — Frozen backbone training')
        print(f'   Epochs to train : {remaining_freeze} (total={FREEZE_EPOCHS})')
        print(f'   Trainable layers: head only (backbone frozen)')
        print(f'   Learning rate   : {FREEZE_LR}')

        # If resuming, load last checkpoint
        if state['epoch_done'] > 0 and os.path.exists(FREEZE_CKPT):
            print(f'\n📥 Loading freeze-phase checkpoint from Drive...')
            model = keras.models.load_model(FREEZE_CKPT)
            model, backbone = build_model()   # rebuild to get backbone reference
            model.load_weights(FREEZE_CKPT)

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=FREEZE_LR),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        freeze_callbacks = make_callbacks(
            'freeze',
            FREEZE_CKPT,
            f'{LOG_DIR}/freeze_phase_log.csv'
        )

        freeze_history = model.fit(
            train_ds,
            epochs=remaining_freeze,
            validation_data=val_ds,
            class_weight=class_weights_dict,
            callbacks=freeze_callbacks,
            verbose=1
        )

        best_freeze_acc = max(freeze_history.history.get('val_accuracy', [0.0]))
        save_training_state('finetune', 0, best_freeze_acc)
        print(f'\n✅ Freeze phase done! Best val_accuracy: {best_freeze_acc*100:.2f}%')
        print(f'💾 Checkpoint saved: {FREEZE_CKPT}')
    else:
        print('⏭️  Freeze phase already complete — skipping to fine-tune')
else:
    print('⏭️  Skipping freeze phase (already completed in previous session)')

In [ ]:
# ─── 6-C: Phase 2 — Full Fine-Tune (Unfreeze ALL layers) ────────────────────
# ANSWER:
#   Unfreeze ALL backbone layers.
#   Use very small LR (1e-5) to avoid catastrophic forgetting.
#   This is where accuracy goes from ~85% → 92–97%.

state = load_training_state()
remaining_finetune = FINETUNE_EPOCHS - state['epoch_done']

print(f'\n🔓 PHASE 2 — Full fine-tune (all layers unfrozen)')
print(f'   Remaining epochs : {remaining_finetune} / {FINETUNE_EPOCHS}')
print(f'   Learning rate    : {FINETUNE_LR}')

# Load best checkpoint from freeze phase (if available)
if os.path.exists(FREEZE_CKPT) and state['epoch_done'] == 0:
    print(f'📥 Loading best freeze-phase weights from Drive...')
    model = keras.models.load_model(FREEZE_CKPT)
elif os.path.exists(BEST_CKPT) and state['epoch_done'] > 0:
    print(f'📥 Resuming fine-tune from last Drive checkpoint...')
    model = keras.models.load_model(BEST_CKPT)

# Unfreeze ALL layers
model.trainable = True

# OVERFITTING PREVENTION ANSWER:
#   1. Very small LR (1e-5) → controlled updates
#   2. class_weight → no bias toward majority class
#   3. EarlyStopping patience=7 → stops before overfitting
#   4. ReduceLROnPlateau → adapts LR automatically
#   5. Dropout 0.40 in head (already built in)
#   6. L2 regularization on Dense layers (already built in)
#   7. Heavy augmentation in training pipeline (Phase 3B + 4A)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=FINETUNE_LR,
        clipnorm=1.0   # gradient clipping → extra stability
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f'✅ All layers unfrozen. Trainable params: {sum([np.prod(v.shape) for v in model.trainable_variables]):,}')

finetune_callbacks = make_callbacks(
    'finetune',
    BEST_CKPT,
    f'{LOG_DIR}/finetune_phase_log.csv'
)

if remaining_finetune > 0:
    finetune_history = model.fit(
        train_ds,
        epochs=remaining_finetune,
        validation_data=val_ds,
        class_weight=class_weights_dict,
        callbacks=finetune_callbacks,
        verbose=1
    )

    best_ft_acc = max(finetune_history.history.get('val_accuracy', [0.0]))
    save_training_state('done', FINETUNE_EPOCHS, best_ft_acc)
    print(f'\n✅ Fine-tune complete! Best val_accuracy: {best_ft_acc*100:.2f}%')
    print(f'💾 Best model saved: {BEST_CKPT}')
else:
    print('✅ Fine-tune already complete from previous session')

In [ ]:
# ─── 6-D: Save final model — .keras + joblib metadata ───────────────────────
# Load best checkpoint
print('📥 Loading best model from Drive checkpoint...')
final_model = keras.models.load_model(BEST_CKPT)

# Save as .keras (Keras v3 native format — recommended)
final_model.save(MODEL_KERAS)
print(f'✅ Model saved as .keras: {MODEL_KERAS}')

# Save model metadata with joblib
# ANSWER: joblib is used to store preprocessing objects & config together
model_metadata = {
    'img_size'       : IMG_SIZE,
    'num_classes'    : NUM_CLASSES,
    'class_names'    : CLASS_NAMES,
    'model_path'     : MODEL_KERAS,
    'architecture'   : 'EfficientNetV2B1',
    'freeze_epochs'  : FREEZE_EPOCHS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'freeze_lr'      : FREEZE_LR,
    'finetune_lr'    : FINETUNE_LR,
    'batch_size'     : BATCH_SIZE,
    'created_at'     : time.strftime('%Y-%m-%d %H:%M:%S')
}
joblib.dump(model_metadata, f'{JOBLIB_DIR}/model_metadata.joblib')
print(f'✅ Metadata saved: {JOBLIB_DIR}/model_metadata.joblib')

# Reload LabelEncoder
le_loaded = joblib.load(f'{JOBLIB_DIR}/label_encoder.joblib')
cw_loaded = joblib.load(f'{JOBLIB_DIR}/class_weights.joblib')
print('✅ joblib objects verified: LabelEncoder and ClassWeights reloaded OK')

## 📊 Phase 7 — Evaluation, Confusion Matrix, Training Plots

In [ ]:
# ─── 7-A: Plot training history from CSV logs on Drive ──────────────────────
# ANSWER: CSVLogger saves every epoch to Drive — plot from CSV (works after resume too)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Training History — EfficientNetV2-B1 DR Classification', fontsize=14, fontweight='bold')

for i, (phase, log_file) in enumerate([
    ('Freeze Phase (Head Only)',  f'{LOG_DIR}/freeze_phase_log.csv'),
    ('Fine-tune Phase (All Layers)', f'{LOG_DIR}/finetune_phase_log.csv')
]):
    if not os.path.exists(log_file):
        continue
    hist_df = pd.read_csv(log_file)

    ax_acc  = axes[0][i]
    ax_loss = axes[1][i]

    ax_acc.plot(hist_df['accuracy'],     label='Train Acc',  linewidth=2, color='royalblue')
    ax_acc.plot(hist_df['val_accuracy'], label='Val Acc',    linewidth=2, color='tomato')
    ax_acc.set_title(f'{phase} — Accuracy', fontweight='bold')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(); ax_acc.grid(alpha=0.3)
    # Add max val_acc annotation
    best_e = hist_df['val_accuracy'].idxmax()
    best_v = hist_df['val_accuracy'].max()
    ax_acc.annotate(f'Best: {best_v*100:.1f}%', xy=(best_e, best_v),
                    xytext=(best_e+0.5, best_v-0.05),
                    arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

    ax_loss.plot(hist_df['loss'],     label='Train Loss', linewidth=2, color='royalblue')
    ax_loss.plot(hist_df['val_loss'], label='Val Loss',   linewidth=2, color='tomato')
    ax_loss.set_title(f'{phase} — Loss', fontweight='bold')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(); ax_loss.grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{DRIVE_BASE}/training_history_plots.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Plot saved: {plot_path}')

In [ ]:
# ─── 7-B: Final evaluation on test set ──────────────────────────────────────
print('🔍 Evaluating on test set...')
test_loss, test_acc = final_model.evaluate(test_ds, verbose=1)
print(f'\n🎯 Test Accuracy : {test_acc*100:.2f}%')
print(f'   Test Loss     : {test_loss:.4f}')

# Get predictions and true labels
y_true, y_pred = [], []
for imgs, lbls in tqdm(test_ds, desc='Predicting'):
    preds = final_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(lbls.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification report
print('\n📋 Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Save report to Drive
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(f'{DRIVE_BASE}/classification_report.csv')
print(f'💾 Report saved: {DRIVE_BASE}/classification_report.csv')

In [ ]:
# ─── 7-C: Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Confusion Matrix — EfficientNetV2-B1 DR Classification', fontsize=13, fontweight='bold')

# Raw counts
disp1 = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp1.plot(ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title('Raw Counts', fontweight='bold')
ax1.tick_params(axis='x', rotation=30)

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(np.round(cm_norm, 2), display_labels=CLASS_NAMES)
disp2.plot(ax=ax2, colorbar=False, cmap='Blues')
ax2.set_title('Normalized (Row %)', fontweight='bold')
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
cm_path = f'{DRIVE_BASE}/confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Confusion matrix saved: {cm_path}')

## ♻️ Phase 8 — Resume Training (Run This After Any Crash)

> **ANSWER: Full resume workflow after Colab disconnects**
> 1. Mount Drive
> 2. Run Phase 0 imports (re-install packages)
> 3. Run this cell — it reads `training_state.json` and loads the last checkpoint automatically
> 4. Continues training from exactly where it stopped

In [ ]:
# ─── 8-A: ONE-CELL RESUME — Run this after any crash ────────────────────────
# This cell reads the JSON state file and continues from the right phase & epoch

print('🔄 Checking training state on Drive...')

if not os.path.exists(RESUME_JSON):
    print('⚠️  No training_state.json found — start from Phase 5 (build model)')
else:
    state = load_training_state()
    print(f'📍 Phase        : {state["phase"]}')
    print(f'📍 Epoch done   : {state["epoch_done"]}')
    print(f'📍 Best val acc : {state["best_val_acc"]*100:.2f}%')
    print(f'📍 Timestamp    : {state.get("timestamp", "unknown")}')

    if state['phase'] == 'done':
        print('\n✅ Training is COMPLETE. Load final model for inference:')
        print(f'   final_model = keras.models.load_model("{MODEL_KERAS}")')
    elif state['phase'] == 'freeze':
        remaining = FREEZE_EPOCHS - state['epoch_done']
        ckpt_to_load = FREEZE_CKPT if os.path.exists(FREEZE_CKPT) else None
        print(f'\n▶️  RESUME: Freeze phase — {remaining} epochs remaining')
        if ckpt_to_load:
            print(f'   Loading: {ckpt_to_load}')
            model = keras.models.load_model(ckpt_to_load)
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=FREEZE_LR),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            resume_callbacks = make_callbacks('freeze', FREEZE_CKPT, f'{LOG_DIR}/freeze_phase_log.csv')
            model.fit(
                train_ds, epochs=remaining, validation_data=val_ds,
                class_weight=class_weights_dict, callbacks=resume_callbacks, verbose=1
            )
            save_training_state('finetune', 0, max(model.history.history.get('val_accuracy', [state['best_val_acc']])))
            print('✅ Freeze phase resumed and completed')
    elif state['phase'] == 'finetune':
        remaining = FINETUNE_EPOCHS - state['epoch_done']
        ckpt_to_load = BEST_CKPT if os.path.exists(BEST_CKPT) else FREEZE_CKPT
        print(f'\n▶️  RESUME: Fine-tune phase — {remaining} epochs remaining')
        if os.path.exists(ckpt_to_load):
            print(f'   Loading: {ckpt_to_load}')
            model = keras.models.load_model(ckpt_to_load)
            model.trainable = True
            model.compile(
                optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR, clipnorm=1.0),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            resume_callbacks = make_callbacks('finetune', BEST_CKPT, f'{LOG_DIR}/finetune_phase_log.csv')
            history = model.fit(
                train_ds, epochs=remaining, validation_data=val_ds,
                class_weight=class_weights_dict, callbacks=resume_callbacks, verbose=1
            )
            best_acc = max(history.history.get('val_accuracy', [state['best_val_acc']]))
            save_training_state('done', FINETUNE_EPOCHS, best_acc)
            model.save(MODEL_KERAS)
            print(f'\n✅ Fine-tune resumed and completed! Best val_accuracy: {best_acc*100:.2f}%')
            print(f'💾 Final model saved: {MODEL_KERAS}')

## 📋 Final Summary

| Question | Answer |
|----------|--------|
| **Images per class?** | Min 1500 real + augment to 2000 |
| **Imbalance fix?** | Class weights + augmentation + CLAHE |
| **Corrupted files?** | PIL verify() scan → auto-delete |
| **Extract zips?** | ThreadPoolExecutor → all parallel |
| **Combine datasets?** | Master CSV with normalized 0-4 labels |
| **Epochs?** | 20 frozen + 40 fine-tune + EarlyStopping(7) |
| **Resolution?** | 384×384 (EfficientNetV2-B1 native) |
| **90-100% accuracy?** | Transfer learning → freeze → unfreeze → CLAHE aug |
| **Overfitting?** | Dropout 0.4 + L2 + EarlyStopping + ReduceLR + Aug |
| **Save to Drive?** | ModelCheckpoint(save_best_only) + CSVLogger |
| **Resume?** | training_state.json + load_model(BEST_CKPT) |
| **joblib?** | LabelEncoder + ClassWeights + Metadata saved |
| **keras file?** | model.save(path.keras) → load_model(path.keras) |

In [ ]:
# ─── Final Summary Print ────────────────────────────────────────────────────
print('=' * 70)
print('  🩺 DR EfficientNetV2-B1 COMPLETE PIPELINE — FINAL REPORT')
print('=' * 70)

# Load state
if os.path.exists(RESUME_JSON):
    state = load_training_state()
    print(f'   Training Phase   : {state["phase"]}')
    print(f'   Best Val Acc     : {state["best_val_acc"]*100:.2f}%')

print(f'\n📁 DRIVE FILES')
for f in [
    MODEL_KERAS,
    BEST_CKPT,
    RESUME_JSON,
    f'{DRIVE_BASE}/master_labels.csv',
    f'{DRIVE_BASE}/balanced_train_labels.csv',
    f'{DRIVE_BASE}/classification_report.csv',
    f'{DRIVE_BASE}/confusion_matrix.png',
    f'{DRIVE_BASE}/training_history_plots.png',
    f'{JOBLIB_DIR}/label_encoder.joblib',
    f'{JOBLIB_DIR}/class_weights.joblib',
    f'{JOBLIB_DIR}/model_metadata.joblib',
    f'{LOG_DIR}/freeze_phase_log.csv',
    f'{LOG_DIR}/finetune_phase_log.csv',
]:
    status = '✅' if os.path.exists(f) else '❌'
    print(f'   {status} {os.path.basename(f)}')

print(f'\n🚀 LOAD MODEL FOR INFERENCE')
print(f'   import tensorflow as tf, joblib')
print(f'   model    = tf.keras.models.load_model("{MODEL_KERAS}")')
print(f'   le       = joblib.load("{JOBLIB_DIR}/label_encoder.joblib")')
print(f'   meta     = joblib.load("{JOBLIB_DIR}/model_metadata.joblib")')
print(f'   img      = tf.image.resize(img, (384, 384)) / 255.0')
print(f'   pred_cls = tf.argmax(model.predict(img[None,...]), axis=1).numpy()[0]')
print(f'   label    = le.inverse_transform([pred_cls])[0]')
print()
print('=' * 70)
print('✅ PIPELINE COMPLETE')
print('=' * 70)